In [1]:
# Regularize Random Forest, tune on walk-forward validation, and audit validation importance.
# Freeze every choice before evaluating the final test set.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import (
    project_paths,
    file_hash,
    probability_metrics,
    load_split,
)
from pathlib import Path
import time
import warnings
import joblib
import platform
import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
import statsmodels.api as sm

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/train_matrix.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
MODELS = project_paths(ROOT).models
REPORTS = project_paths(ROOT).reports / "06_random_forest_reporting"
SEED = 42
N_ESTIMATORS = 200
N_JOBS = 1
MAX_DEPTHS = [2, 4]
MIN_LEAF_SIZES = [10, 25]
MIN_SPLITS = [20, 50]
MAX_FEATURES = [0.1, 0.25]
MAX_SAMPLES = [0.5, 0.75]
CLASS_WEIGHTS = {"balanced_subsample": "balanced_subsample"}
PERMUTATION_REPEATS = 3
BOOTSTRAPS = 20
BOOTSTRAP_SEED = 2401
input_hashes = {}

artifact_path = MODELS / "preprocessing_pipeline.joblib"
input_hashes[artifact_path] = file_hash(artifact_path)
preprocessing = joblib.load(artifact_path)
schema = preprocessing.project_schema_
if schema.get("specification") != "B" or schema.get("fit_split") != "train":
    raise ValueError("Random Forest requires training-fitted Spec B preprocessing")
FEATURES, ID_COLUMNS = (
    schema["matrix_feature_columns"],
    schema["matrix_metadata_columns"],
)


def make_forest(depth, leaf, split, features, samples, weight):
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=depth,
        min_samples_leaf=leaf,
        min_samples_split=split,
        max_features=features,
        max_samples=samples,
        class_weight=weight,
        bootstrap=True,
        oob_score=False,
        random_state=SEED,
        n_jobs=N_JOBS,
    )


def calibration_fit(y, p):
    q = np.clip(p, 1e-6, 1 - 1e-6)
    z = np.log(q / (1 - q))
    try:
        fit = sm.GLM(
            y, sm.add_constant(z, has_constant="add"), family=sm.families.Binomial()
        ).fit(maxiter=100, disp=0)
        return float(fit.params[0]), float(fit.params[1])
    except Exception:
        return np.nan, np.nan

In [3]:
# Tune stronger tree, leaf, split, feature, sample, and class-weight constraints.
fold_ids_path = PROCESSED / "walk_forward_fold_ids.csv"
input_hashes[fold_ids_path] = file_hash(fold_ids_path)
manifest_cols = [
    "fold",
    "split",
    "company_id",
    "prediction_date",
    "metadata_event_id",
    "company_group_id",
    "metadata_outcome_end_date",
]
manifest = pd.read_csv(
    fold_ids_path,
    usecols=manifest_cols,
    dtype={
        "company_id": "string",
        "metadata_event_id": "string",
        "company_group_id": "string",
    },
)
manifest["prediction_date"] = pd.to_datetime(manifest.prediction_date, errors="raise")
manifest["metadata_outcome_end_date"] = pd.to_datetime(
    manifest.metadata_outcome_end_date, errors="raise"
)
fold_names = sorted(manifest.loc[manifest.fold.ne("final_test"), "fold"].unique())
fold_data = {}
for fold in fold_names:
    members = manifest.loc[manifest.fold.eq(fold)]
    train_ids = members.loc[members.split.eq("train")]
    val_ids = members.loc[members.split.eq("validation")]
    if train_ids.empty or val_ids.empty:
        raise ValueError(f"{fold}: missing training/validation rows")
    if set(train_ids.company_group_id).intersection(val_ids.company_group_id):
        raise ValueError(f"{fold}: company group crosses split")
    if not train_ids.metadata_outcome_end_date.max() < val_ids.prediction_date.min():
        raise ValueError(f"{fold}: outcome window reaches validation")
    pipeline_path = (
        MODELS / "walk_forward" / f"{fold}_spec_b_preprocessing_pipeline.joblib"
    )
    train_path = PROCESSED / "walk_forward" / fold / "spec_b" / "train_matrix.csv"
    val_path = PROCESSED / "walk_forward" / fold / "spec_b" / "validation_matrix.csv"
    for path in [pipeline_path, train_path, val_path]:
        input_hashes[path] = file_hash(path)
    fold_pipeline = joblib.load(pipeline_path)
    fold_schema = fold_pipeline.project_schema_
    if (
        fold_schema.get("fit_split") != f"{fold}/train"
        or fold_schema.get("specification") != "B"
    ):
        raise ValueError(f"{fold}: wrong fold pipeline")
    if fold_schema["training_event_ids"] != train_ids.metadata_event_id.tolist():
        raise ValueError(f"{fold}: pipeline training membership mismatch")
    train_frame = pd.read_csv(
        train_path, dtype={"company_id": "string", "metadata_event_id": "string"}
    )
    val_frame = pd.read_csv(
        val_path, dtype={"company_id": "string", "metadata_event_id": "string"}
    )
    fold_features = fold_schema["matrix_feature_columns"]
    if (
        train_frame.columns.tolist() != ID_COLUMNS + fold_features
        or val_frame.columns.tolist() != ID_COLUMNS + fold_features
    ):
        raise ValueError(f"{fold}: matrix schema mismatch")
    if (
        train_frame.metadata_event_id.tolist() != train_ids.metadata_event_id.tolist()
        or val_frame.metadata_event_id.tolist() != val_ids.metadata_event_id.tolist()
    ):
        raise ValueError(f"{fold}: matrix membership/order mismatch")
    selector = VarianceThreshold(0.0)
    Xtr = sparse.csr_matrix(
        selector.fit_transform(train_frame[fold_features].to_numpy(dtype=float))
    )
    Xva = sparse.csr_matrix(
        selector.transform(val_frame[fold_features].to_numpy(dtype=float))
    )
    fold_data[fold] = (
        train_frame,
        val_frame,
        fold_features,
        selector,
        Xtr,
        Xva,
        train_frame.target.to_numpy(dtype=int),
        val_frame.target.to_numpy(dtype=int),
    )

fold_rows = []
oof_by_candidate = {}
for weight_name, weight in CLASS_WEIGHTS.items():
    for depth in MAX_DEPTHS:
        for leaf in MIN_LEAF_SIZES:
            for split in MIN_SPLITS:
                for max_features in MAX_FEATURES:
                    for max_samples in MAX_SAMPLES:
                        candidate = f"{weight_name}_d{depth}_leaf{leaf}_split{split}_mf{max_features}_ms{max_samples}"
                        predictions_by_fold = {}
                        for fold, (
                            tr,
                            va,
                            cols,
                            selector,
                            Xtr,
                            Xva,
                            ytr,
                            yva,
                        ) in fold_data.items():
                            forest = make_forest(
                                depth, leaf, split, max_features, max_samples, weight
                            )
                            started = time.perf_counter()
                            with warnings.catch_warnings(record=True) as caught:
                                warnings.simplefilter("always")
                                forest.fit(Xtr, ytr)
                            prob = forest.predict_proba(Xva)[:, 1]
                            eligible = bool(np.isfinite(prob).all())
                            fold_rows.append(
                                {
                                    "candidate_id": candidate,
                                    "fold": fold,
                                    "n_estimators": N_ESTIMATORS,
                                    "max_depth": depth,
                                    "min_samples_leaf": leaf,
                                    "min_samples_split": split,
                                    "max_features": str(max_features),
                                    "max_samples": max_samples,
                                    "class_weight": weight_name,
                                    "eligible": eligible,
                                    "mean_tree_depth": float(
                                        np.mean(
                                            [
                                                tree.tree_.max_depth
                                                for tree in forest.estimators_
                                            ]
                                        )
                                    ),
                                    "validation_pr_auc": average_precision_score(
                                        yva, prob
                                    ),
                                    "validation_roc_auc": roc_auc_score(yva, prob),
                                    "validation_brier_score": brier_score_loss(
                                        yva, prob
                                    ),
                                    "elapsed_seconds": time.perf_counter() - started,
                                    "warnings": "; ".join(
                                        sorted({str(w.message) for w in caught})
                                    ),
                                }
                            )
                            if eligible:
                                predictions_by_fold[fold] = pd.DataFrame(
                                    {
                                        "company_id": va.company_id,
                                        "prediction_date": va.prediction_date,
                                        "metadata_event_id": va.metadata_event_id,
                                        "fold": fold,
                                        "target": yva,
                                        "bankruptcy_probability": prob,
                                    }
                                )
                        if len(predictions_by_fold) == len(fold_data):
                            oof_by_candidate[candidate] = predictions_by_fold

fold_tuning = pd.DataFrame(fold_rows)
tuning = fold_tuning.groupby(
    [
        "candidate_id",
        "n_estimators",
        "max_depth",
        "min_samples_leaf",
        "min_samples_split",
        "max_features",
        "max_samples",
        "class_weight",
    ],
    as_index=False,
).agg(
    eligible=("eligible", "all"),
    folds_completed=("fold", "nunique"),
    mean_validation_pr_auc=("validation_pr_auc", "mean"),
    mean_validation_roc_auc=("validation_roc_auc", "mean"),
    mean_validation_brier_score=("validation_brier_score", "mean"),
)
tuning = (
    tuning.loc[tuning.eligible]
    .sort_values(
        [
            "mean_validation_pr_auc",
            "mean_validation_roc_auc",
            "mean_validation_brier_score",
            "candidate_id",
        ],
        ascending=[False, False, True, True],
        kind="stable",
    )
    .reset_index(drop=True)
)
if tuning.empty:
    raise RuntimeError("No eligible Random Forest candidate")
best = tuning.iloc[0]
oof = pd.concat(list(oof_by_candidate[best.candidate_id].values()), ignore_index=True)
if oof.metadata_event_id.duplicated().any() or oof.target.nunique() != 2:
    raise ValueError(
        "Pooled OOF predictions must contain unique events and both classes"
    )
oof = oof.sort_values(["prediction_date", "company_id"], kind="stable").reset_index(
    drop=True
)

In [4]:
# Refit the selected forest within each fold and measure validation permutation importance.
importance_rows = []
bootstrap_rows = []
oof_selected = []
rng = np.random.default_rng(BOOTSTRAP_SEED)
for fold, (tr, va, cols, selector, Xtr, Xva, ytr, yva) in fold_data.items():
    forest = make_forest(
        best.max_depth,
        int(best.min_samples_leaf),
        int(best.min_samples_split),
        float(best.max_features) if str(best.max_features) != "sqrt" else "sqrt",
        float(best.max_samples),
        CLASS_WEIGHTS[best.class_weight],
    )
    forest.fit(Xtr, ytr)
    base_probability = forest.predict_proba(Xva)[:, 1]
    oof_selected.append(
        pd.DataFrame(
            {"metadata_event_id": va.metadata_event_id, "probability": base_probability}
        )
    )
    base_ap = average_precision_score(yva, base_probability)
    active_names = np.asarray(cols)[selector.get_support()].tolist()
    dense = Xva.toarray()
    n, p = dense.shape
    permuted_probabilities = {name: [] for name in active_names}
    # Batch feature permutations to reduce forest prediction overhead.
    batch_features = max(1, 10)
    for start in range(0, p, batch_features):
        feature_batch = list(range(start, min(start + batch_features, p)))
        matrices = []
        keys = []
        for j in feature_batch:
            for repeat in range(PERMUTATION_REPEATS):
                changed = dense.copy()
                changed[:, j] = rng.permutation(changed[:, j])
                matrices.append(changed)
                keys.append((active_names[j], repeat))
        predicted = forest.predict_proba(np.vstack(matrices))[:, 1].reshape(
            len(keys), n
        )
        for key, prob in zip(keys, predicted):
            permuted_probabilities[key[0]].append(prob)
    case_indices = np.flatnonzero(yva == 1)
    control_indices = np.flatnonzero(yva == 0)
    bootstrap_importance = {name: [] for name in active_names}
    bootstrap_top = {name: 0 for name in active_names}
    for b in range(BOOTSTRAPS):
        sample = np.r_[
            rng.choice(case_indices, len(case_indices), replace=True),
            rng.choice(control_indices, len(control_indices), replace=True),
        ]
        base_score = average_precision_score(yva[sample], base_probability[sample])
        scores = {}
        for name in active_names:
            perm_scores = [
                average_precision_score(yva[sample], prob[sample])
                for prob in permuted_probabilities[name]
            ]
            scores[name] = base_score - float(np.mean(perm_scores))
            bootstrap_importance[name].append(scores[name])
        top = sorted(scores, key=scores.get, reverse=True)[:10]
        for name in top:
            bootstrap_top[name] += 1
    for j, name in enumerate(active_names):
        drops = [
            base_ap - average_precision_score(yva, prob)
            for prob in permuted_probabilities[name]
        ]
        bootstrap_values = np.asarray(bootstrap_importance[name])
        importance_rows.append(
            {
                "fold": fold,
                "feature": name,
                "validation_average_precision": base_ap,
                "permutation_importance_mean": float(np.mean(drops)),
                "permutation_importance_sd": float(np.std(drops, ddof=1)),
                "permutation_repeats": PERMUTATION_REPEATS,
            }
        )
        bootstrap_rows.append(
            {
                "fold": fold,
                "feature": name,
                "bootstrap_mean_importance": float(bootstrap_values.mean()),
                "bootstrap_sd": float(bootstrap_values.std(ddof=1)),
                "bootstrap_positive_fraction": float((bootstrap_values > 0).mean()),
                "bootstrap_top10_frequency": bootstrap_top[name] / BOOTSTRAPS,
                "bootstrap_replicates": BOOTSTRAPS,
            }
        )

permutation_by_fold = pd.DataFrame(importance_rows)
bootstrap_by_fold = pd.DataFrame(bootstrap_rows)
permutation_importance = permutation_by_fold.groupby("feature", as_index=False).agg(
    mean_permutation_importance=("permutation_importance_mean", "mean"),
    fold_importance_sd=("permutation_importance_mean", "std"),
    mean_permutation_sd=("permutation_importance_sd", "mean"),
    validation_folds=("fold", "nunique"),
)
stability = bootstrap_by_fold.groupby("feature", as_index=False).agg(
    bootstrap_mean_importance=("bootstrap_mean_importance", "mean"),
    bootstrap_sd=("bootstrap_sd", "mean"),
    bootstrap_positive_fraction=("bootstrap_positive_fraction", "mean"),
    bootstrap_top10_frequency=("bootstrap_top10_frequency", "mean"),
    validation_folds=("fold", "nunique"),
)
permutation_importance = permutation_importance.merge(
    stability, on=["feature", "validation_folds"], how="left"
)
permutation_importance = permutation_importance.sort_values(
    "mean_permutation_importance", ascending=False, kind="stable"
).reset_index(drop=True)

# Select the classification threshold using only pooled out-of-fold validation probabilities.
y_oof = oof.target.to_numpy(dtype=int)
p_oof = oof.bankruptcy_probability.to_numpy(dtype=float)
thresholds = np.unique(np.r_[0.0, 0.5, p_oof, np.nextafter(p_oof.max(), np.inf)])
threshold_rows = pd.DataFrame(
    [probability_metrics(y_oof, p_oof, t) for t in thresholds]
)
threshold_rows["distance_from_half"] = (threshold_rows.threshold - 0.5).abs()
threshold_rows = threshold_rows.sort_values(
    [
        "balanced_accuracy",
        "sensitivity",
        "specificity",
        "distance_from_half",
        "threshold",
    ],
    ascending=[False, False, False, True, False],
    kind="stable",
).reset_index(drop=True)
OPERATING_THRESHOLD = float(threshold_rows.iloc[0].threshold)

configuration = {
    "selected_candidate": best.candidate_id,
    "selection_metric": "mean walk-forward validation PR-AUC",
    "mean_validation_pr_auc": float(best.mean_validation_pr_auc),
    "operating_threshold": OPERATING_THRESHOLD,
    "threshold_source": "pooled out-of-fold validation predictions",
    "threshold_metric": "balanced accuracy with sensitivity/specificity tie-breaks",
    "n_estimators": N_ESTIMATORS,
    "max_depth": None if pd.isna(best.max_depth) else int(best.max_depth),
    "min_samples_leaf": int(best.min_samples_leaf),
    "min_samples_split": int(best.min_samples_split),
    "max_features": str(best.max_features),
    "max_samples": float(best.max_samples),
    "class_weight": best.class_weight,
    "walk_forward_folds": fold_names,
    "seed": SEED,
    "bootstrap": True,
    "preprocessing_sha256": input_hashes[artifact_path],
}

# Add serial Gradient Boosting as an optional sensitivity check for the remaining forest fit gap.
GB_GRID = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [1, 2],
    "min_samples_leaf": [10, 20],
}
gb_rows = []
gb_oof_candidates = {}
for n_estimators in GB_GRID["n_estimators"]:
    for learning_rate in GB_GRID["learning_rate"]:
        for depth in GB_GRID["max_depth"]:
            for min_leaf in GB_GRID["min_samples_leaf"]:
                candidate = (
                    f"trees{n_estimators}_lr{learning_rate}_depth{depth}_leaf{min_leaf}"
                )
                candidate_predictions = {}
                for fold, (
                    tr,
                    va,
                    cols,
                    fold_selector,
                    Xtr,
                    Xva,
                    ytr,
                    yva,
                ) in fold_data.items():
                    gb = GradientBoostingClassifier(
                        n_estimators=n_estimators,
                        learning_rate=learning_rate,
                        max_depth=depth,
                        min_samples_leaf=min_leaf,
                        max_features=0.25,
                        subsample=0.8,
                        random_state=SEED,
                    )
                    gb.fit(
                        Xtr.toarray(),
                        ytr,
                        sample_weight=compute_sample_weight("balanced", ytr),
                    )
                    prob = gb.predict_proba(Xva.toarray())[:, 1]
                    gb_rows.append(
                        {
                            "candidate_id": candidate,
                            "fold": fold,
                            "n_estimators": n_estimators,
                            "learning_rate": learning_rate,
                            "max_depth": depth,
                            "min_samples_leaf": min_leaf,
                            "validation_pr_auc": average_precision_score(yva, prob),
                            "validation_roc_auc": roc_auc_score(yva, prob),
                            "validation_brier_score": brier_score_loss(yva, prob),
                        }
                    )
                    candidate_predictions[fold] = pd.DataFrame(
                        {
                            "company_id": va.company_id,
                            "prediction_date": va.prediction_date,
                            "metadata_event_id": va.metadata_event_id,
                            "fold": fold,
                            "target": yva,
                            "bankruptcy_probability": prob,
                        }
                    )
                if len(candidate_predictions) == len(fold_data):
                    gb_oof_candidates[candidate] = candidate_predictions

gb_fold_tuning = pd.DataFrame(gb_rows)
gb_tuning = gb_fold_tuning.groupby(
    ["candidate_id", "n_estimators", "learning_rate", "max_depth", "min_samples_leaf"],
    as_index=False,
).agg(
    mean_validation_pr_auc=("validation_pr_auc", "mean"),
    mean_validation_roc_auc=("validation_roc_auc", "mean"),
    mean_validation_brier_score=("validation_brier_score", "mean"),
)
gb_tuning = gb_tuning.sort_values(
    [
        "mean_validation_pr_auc",
        "mean_validation_roc_auc",
        "mean_validation_brier_score",
        "candidate_id",
    ],
    ascending=[False, False, True, True],
    kind="stable",
).reset_index(drop=True)
gb_best = gb_tuning.iloc[0]
gb_oof = pd.concat(
    list(gb_oof_candidates[gb_best.candidate_id].values()), ignore_index=True
)
gb_y_oof = gb_oof.target.to_numpy(dtype=int)
gb_p_oof = gb_oof.bankruptcy_probability.to_numpy(dtype=float)
gb_thresholds = np.unique(
    np.r_[0.0, 0.5, gb_p_oof, np.nextafter(gb_p_oof.max(), np.inf)]
)
gb_threshold_rows = pd.DataFrame(
    [probability_metrics(gb_y_oof, gb_p_oof, t) for t in gb_thresholds]
)
gb_threshold_rows["distance_from_half"] = (gb_threshold_rows.threshold - 0.5).abs()
gb_threshold_rows = gb_threshold_rows.sort_values(
    [
        "balanced_accuracy",
        "sensitivity",
        "specificity",
        "distance_from_half",
        "threshold",
    ],
    ascending=[False, False, False, True, False],
    kind="stable",
).reset_index(drop=True)
GB_THRESHOLD = float(gb_threshold_rows.iloc[0].threshold)
gb_configuration = {
    "role": "optional sensitivity model",
    "selected_candidate": gb_best.candidate_id,
    "selection_metric": "mean walk-forward validation PR-AUC",
    "mean_validation_pr_auc": float(gb_best.mean_validation_pr_auc),
    "operating_threshold": GB_THRESHOLD,
    "threshold_source": "pooled out-of-fold validation predictions",
    "n_estimators": int(gb_best.n_estimators),
    "learning_rate": float(gb_best.learning_rate),
    "max_depth": int(gb_best.max_depth),
    "min_samples_leaf": int(gb_best.min_samples_leaf),
    "max_features": 0.25,
    "subsample": 0.8,
    "class_weight": "balanced sample weights",
    "walk_forward_folds": fold_names,
}

# Fit the selected configuration on the static training split, then open the test set.
train_path = PROCESSED / "train_matrix.csv"
train_ids_path = PROCESSED / "train_ids.csv"
for path in [train_path, train_ids_path]:
    input_hashes[path] = file_hash(path)
train = load_split("train", PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
selector = VarianceThreshold(0.0)
X_train = selector.fit_transform(train[FEATURES].to_numpy(dtype=float))
y_train = train.target.to_numpy(dtype=int)
final_model = make_forest(
    configuration["max_depth"],
    configuration["min_samples_leaf"],
    configuration["min_samples_split"],
    float(configuration["max_features"])
    if configuration["max_features"] != "sqrt"
    else "sqrt",
    configuration["max_samples"],
    CLASS_WEIGHTS[configuration["class_weight"]],
)
final_model.fit(X_train, y_train)
gradient_model = GradientBoostingClassifier(
    n_estimators=gb_configuration["n_estimators"],
    learning_rate=gb_configuration["learning_rate"],
    max_depth=gb_configuration["max_depth"],
    min_samples_leaf=gb_configuration["min_samples_leaf"],
    max_features=gb_configuration["max_features"],
    subsample=gb_configuration["subsample"],
    random_state=SEED,
)
gradient_model.fit(
    X_train, y_train, sample_weight=compute_sample_weight("balanced", y_train)
)
test = load_split("test", PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
X_test = selector.transform(test[FEATURES].to_numpy(dtype=float))
y_test = test.target.to_numpy(dtype=int)
p_test = final_model.predict_proba(X_test)[:, 1]
gb_p_test = gradient_model.predict_proba(X_test)[:, 1]

In [5]:
# Compare with Elastic Net only after the forest settings and test evaluation are frozen.
def metric_row(name, y, p, threshold):
    values = probability_metrics(y, p, threshold)
    clipped = np.clip(p, 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped))
    try:
        fit = sm.GLM(
            y,
            sm.add_constant(logits, has_constant="add"),
            family=sm.families.Binomial(),
        ).fit(maxiter=100, disp=0)
        intercept, slope = float(fit.params[0]), float(fit.params[1])
    except Exception:
        intercept, slope = np.nan, np.nan
    return {
        "model": name,
        **values,
        "recall": values["sensitivity"],
        "precision": values["tp"] / (values["tp"] + values["fp"])
        if values["tp"] + values["fp"]
        else 0.0,
        "calibration_intercept": intercept,
        "calibration_slope": slope,
    }


oof_metrics = metric_row(
    "Random Forest OOF validation", y_oof, p_oof, OPERATING_THRESHOLD
)
test_metrics = metric_row(
    "Random Forest final test", y_test, p_test, OPERATING_THRESHOLD
)
p_train = final_model.predict_proba(X_train)[:, 1]
training_diagnostic = metric_row(
    "Random Forest training diagnostic", y_train, p_train, OPERATING_THRESHOLD
)
gb_metrics = pd.DataFrame(
    [
        metric_row(
            "Gradient Boosting sensitivity OOF", gb_y_oof, gb_p_oof, GB_THRESHOLD
        ),
        metric_row(
            "Gradient Boosting sensitivity test", y_test, gb_p_test, GB_THRESHOLD
        ),
    ]
)
metrics = pd.concat(
    [pd.DataFrame([oof_metrics, test_metrics]), gb_metrics], ignore_index=True
)
elastic_artifact_path = MODELS / "elastic_net_logistic.joblib"
elastic_predictions_path = (
    project_paths(ROOT).reports
    / "05_elastic_net_reporting"
    / "elastic_net_predictions.csv"
)
for path in [elastic_artifact_path, elastic_predictions_path]:
    input_hashes[path] = file_hash(path)
elastic_threshold = joblib.load(elastic_artifact_path).project_model_[
    "operating_threshold"
]
elastic_predictions = pd.read_csv(
    elastic_predictions_path,
    dtype={"company_id": "string", "metadata_event_id": "string"},
)
elastic_oof = elastic_predictions.loc[
    elastic_predictions.prediction_split.eq("walk_forward_oof_validation")
].copy()
elastic_test = elastic_predictions.loc[
    elastic_predictions.prediction_split.eq("final_test")
].copy()
if set(elastic_oof.metadata_event_id) != set(oof.metadata_event_id) or set(
    elastic_test.metadata_event_id
) != set(test.metadata_event_id):
    raise ValueError("Elastic Net and Random Forest evaluation memberships differ")
elastic_oof = elastic_oof.set_index("metadata_event_id").loc[oof.metadata_event_id]
elastic_test = elastic_test.set_index("metadata_event_id").loc[test.metadata_event_id]
elastic_metrics = pd.DataFrame(
    [
        metric_row(
            "Elastic Net OOF validation",
            elastic_oof.target.to_numpy(dtype=int),
            elastic_oof.prediction_probability.to_numpy(dtype=float),
            elastic_threshold,
        ),
        metric_row(
            "Elastic Net final test",
            elastic_test.target.to_numpy(dtype=int),
            elastic_test.prediction_probability.to_numpy(dtype=float),
            elastic_threshold,
        ),
    ]
)
comparison = pd.concat([metrics, elastic_metrics], ignore_index=True)

# Preserve predictions for both validation and final test cohorts.
oof["prediction_split"] = "walk_forward_oof_validation"
oof["predicted_target"] = (
    oof.bankruptcy_probability.to_numpy() >= OPERATING_THRESHOLD
).astype(int)
test_predictions = test[ID_COLUMNS].copy()
test_predictions["bankruptcy_probability"] = p_test
test_predictions["predicted_target"] = (p_test >= OPERATING_THRESHOLD).astype(int)
test_predictions["prediction_split"] = "final_test"
rf_predictions = pd.concat([oof, test_predictions], ignore_index=True)
gb_oof["prediction_split"] = "walk_forward_oof_validation"
gb_oof["predicted_target"] = (gb_p_oof >= GB_THRESHOLD).astype(int)
gb_test_predictions = test[ID_COLUMNS].copy()
gb_test_predictions["bankruptcy_probability"] = gb_p_test
gb_test_predictions["predicted_target"] = (gb_p_test >= GB_THRESHOLD).astype(int)
gb_test_predictions["prediction_split"] = "final_test"
gb_predictions = pd.concat([gb_oof, gb_test_predictions], ignore_index=True)

final_pipeline = Pipeline(
    [
        ("preprocessing", preprocessing),
        ("training_constant_selector", selector),
        ("classifier", final_model),
    ]
)
active = np.asarray(FEATURES)[selector.get_support()].tolist()
configuration.update(
    {
        "fit_split": "static training only",
        "active_predictors": len(active),
        "training_event_ids": train.metadata_event_id.tolist(),
        "test_event_ids": test.metadata_event_id.tolist(),
        "versions": {
            "python": platform.python_version(),
            "sklearn": sklearn.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
    }
)
final_pipeline.project_model_ = dict(configuration)
final_pipeline.project_model_["matrix_feature_columns"] = FEATURES
final_pipeline.project_model_["active_feature_columns"] = active

In [6]:
# Save the frozen forest and validation/test audit files.
for path in [
    fold_ids_path,
    artifact_path,
    PROCESSED / "train_matrix.csv",
    PROCESSED / "train_ids.csv",
    PROCESSED / "test_matrix.csv",
    PROCESSED / "test_ids.csv",
    elastic_predictions_path,
    elastic_artifact_path,
]:
    if path not in input_hashes:
        input_hashes[path] = file_hash(path)
final_pipeline.project_model_["input_sha256"] = {
    str(path.relative_to(ROOT)): digest for path, digest in input_hashes.items()
}
joblib.dump(final_pipeline, MODELS / "random_forest.joblib", compress=3)
gb_pipeline = Pipeline(
    [
        ("preprocessing", preprocessing),
        ("training_constant_selector", selector),
        ("classifier", gradient_model),
    ]
)
gb_pipeline.project_model_ = {
    **gb_configuration,
    "matrix_feature_columns": FEATURES,
    "active_feature_columns": active,
    "fit_split": "static training only",
    "test_event_ids": test.metadata_event_id.tolist(),
}
joblib.dump(gb_pipeline, MODELS / "gradient_boosting_sensitivity.joblib", compress=3)

# Run the stage analysis after the notebook has saved its model outputs.
import runpy

analysis = runpy.run_path(ROOT / "analysis" / "06_random_forest_reporting.py")
analysis["generate_report"](globals())